# NEXUS RGB — pixel-skill distillation

Renders frames offline (standard MuJoCo EGL renderer, no MJWarp), then behavior-clones the project's `VisionSkillActor` to control CartpoleBalance from 64x64 pixels. Produces: training loss, held-out pred-vs-teacher, teacher video, and a closed-loop control video + balance metric.

Use a **GPU (T4)** runtime.

In [ ]:
import os
ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(ICD):
    os.makedirs(os.path.dirname(ICD), exist_ok=True)
    open(ICD, 'w').write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'

import os
REPO_URL = 'https://github.com/Tornadosky/nexus_project.git'
BRANCH   = 'rosela'
%cd /content
if not os.path.exists('/content/nexus_project'):
    !git clone {REPO_URL}
%cd /content/nexus_project
!git checkout {BRANCH} && git pull -q origin {BRANCH} || echo '(using local checkout)'
%cd /content/nexus_project/nexus_continuous_control
print('cwd:', os.getcwd())

In [ ]:
# impl='jax' below avoids warp, so no MJWarp/GraphMode issue.
!pip install -q -e ".[dev,analysis,playground]"
!pip install -q mediapy
print('install done')

In [ ]:
import os, numpy as np, jax, jax.numpy as jnp
import mujoco
from mujoco_playground import registry
from nexus_continuous.vision import VisionSkillActor, RGBEncoder
print('jax', jax.__version__, jax.devices())

cfg = registry.get_default_config('CartpoleBalance')
cfg.impl = 'jax'                      # avoids warp entirely
penv = registry.load('CartpoleBalance', cfg)
model = getattr(penv, 'mj_model', None) or getattr(penv, '_mj_model')
data = mujoco.MjData(model); mujoco.mj_forward(model, data)
with mujoco.Renderer(model, height=64, width=64) as r:
    r.update_scene(data, camera=0); frame = r.render()
print('render smoke OK ->', frame.shape, frame.dtype, '| nu=', model.nu)

## Step 1 — Teacher rollout + offline render
A PD balance controller is rolled out in raw MuJoCo; each step is rendered to 64x64. Noise + resets-on-fall give dataset variety (the teacher only needs to define a state->action mapping).

In [ ]:
import matplotlib.pyplot as plt

SLIDER, HINGE = 0, 1            # qpos/qvel indices (slider=0, hinge=1)
rng = np.random.default_rng(0)

def teacher_action(d):
    angle, angvel = d.qpos[HINGE], d.qvel[HINGE]
    cart,  cartvel = d.qpos[SLIDER], d.qvel[SLIDER]
    u = -(8.0*angle + 1.5*angvel + 0.4*cart + 0.8*cartvel) + 0.3*rng.standard_normal()
    return float(np.clip(u, -1.0, 1.0))

def reset(d):
    mujoco.mj_resetData(model, d); d.qpos[HINGE] = 0.05*rng.standard_normal(); mujoco.mj_forward(model, d)

T = 3000
data = mujoco.MjData(model); reset(data)
frames = np.empty((T, 64, 64, 3), np.uint8); acts = np.empty((T, 1), np.float32)
try: cam = 0; mujoco.Renderer(model, 64, 64).update_scene(data, camera=cam)
except Exception: cam = -1   # free camera fallback

with mujoco.Renderer(model, height=64, width=64) as r:
    for t in range(T):
        u = teacher_action(data); data.ctrl[:] = u; mujoco.mj_step(model, data)
        r.update_scene(data, camera=cam); frames[t] = r.render(); acts[t, 0] = u
        if abs(data.qpos[HINGE]) > 1.0: reset(data)

print('rollout:', frames.shape, '| actions range [%.2f, %.2f]' % (acts.min(), acts.max()), '| camera', cam)
fig, ax = plt.subplots(1, 6, figsize=(12, 2.2))
for i, a in enumerate(ax): a.imshow(frames[i*400]); a.axis('off'); a.set_title(f't={i*400}')
plt.suptitle('Offline-rendered teacher rollout'); plt.show()

In [ ]:
import mediapy as media   # teacher rollout video
media.show_video(frames[::2], fps=25)

## Step 2 — Build the pixel dataset
Grayscale, centered to ~[-0.5, 0.5], 3-frame stack (velocity recoverable); target = teacher action.

In [ ]:
g = frames.mean(-1).astype(np.float32) / 255.0 - 0.5
X = np.stack([g[i-2:i+1] for i in range(2, T)], 0).transpose(0, 2, 3, 1)   # [T-2,64,64,3]
Y = acts[2:]
n = len(X); ntr = int(0.9 * n)
Xtr, Ytr, Xval, Yval = X[:ntr], Y[:ntr], X[ntr:], Y[ntr:]
print('dataset:', X.shape, '| train', Xtr.shape[0], 'val', Xval.shape[0])

## Step 3 — Behavior-clone the `VisionSkillActor` (pixels -> action)

In [ ]:
import optax

actor = VisionSkillActor(action_dim=1, action_scale=jnp.ones(1), action_bias=jnp.zeros(1),
                         hidden_sizes=(128, 128), embedding_dim=64)
empty = lambda b: jnp.zeros((b, 0), jnp.float32)   # width-0 proprio (pixels-only)
params = actor.init(jax.random.PRNGKey(0), jnp.asarray(Xtr[:1]), empty(1))['params']
opt = optax.adam(3e-4); opt_state = opt.init(params)

def loss_fn(p, x, y):
    return jnp.mean((actor.apply({'params': p}, x, empty(x.shape[0])) - y) ** 2)

@jax.jit
def train_step(p, st, x, y):
    l, gr = jax.value_and_grad(loss_fn)(p, x, y)
    upd, st = opt.update(gr, st, p); return optax.apply_updates(p, upd), st, l

Xtr_j, Ytr_j = jnp.asarray(Xtr), jnp.asarray(Ytr)
bs, losses = 128, []
for epoch in range(40):
    perm = np.random.permutation(len(Xtr_j)); ep = []
    for i in range(0, len(perm) - bs, bs):
        idx = perm[i:i+bs]
        params, opt_state, l = train_step(params, opt_state, Xtr_j[idx], Ytr_j[idx]); ep.append(float(l))
    losses.append(np.mean(ep))
    if epoch % 5 == 0: print(f'epoch {epoch:2d}  train MSE {losses[-1]:.4f}')

plt.figure(figsize=(6,3)); plt.plot(losses); plt.xlabel('epoch'); plt.ylabel('train MSE')
plt.title('Distillation: VisionSkillActor learning from pixels'); plt.grid(alpha=0.3); plt.show()

## Step 4 — Evaluate (open-loop imitation accuracy)

In [ ]:
pred_val = np.asarray(actor.apply({'params': params}, jnp.asarray(Xval), empty(len(Xval))))
val_mse = float(np.mean((pred_val - Yval) ** 2))
corr = float(np.corrcoef(pred_val[:, 0], Yval[:, 0])[0, 1])
print(f'held-out MSE: {val_mse:.4f}  (teacher var {Yval.var():.4f}) | pred-vs-teacher corr: {corr:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].scatter(Yval[:, 0], pred_val[:, 0], s=4, alpha=0.3)
lim = [-1.1, 1.1]; ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('teacher'); ax[0].set_ylabel('from pixels'); ax[0].set_title('pred vs teacher')
ax[1].plot(Yval[:120, 0], label='teacher'); ax[1].plot(pred_val[:120, 0], label='from pixels', alpha=0.8)
ax[1].set_xlabel('val step'); ax[1].legend(); ax[1].set_title('action trace')
plt.tight_layout(); plt.show()

## Step 5 — Closed-loop control + video (the real metric)
Rolls the trained policy out in the env (render -> policy -> action -> step): a video of it controlling the cart-pole from pixels, plus how many of 250 steps it keeps the pole upright.

In [ ]:
data = mujoco.MjData(model); reset(data)
buf, cl_frames, up = [], [], 0
T_eval = 250
with mujoco.Renderer(model, height=64, width=64) as r:
    for t in range(T_eval):
        r.update_scene(data, camera=cam); rgb = r.render(); cl_frames.append(rgb.copy())
        buf.append(rgb.mean(-1).astype(np.float32) / 255.0 - 0.5); buf = buf[-3:]
        if len(buf) == 3:
            stack = np.stack(buf, -1)[None]
            u = float(np.asarray(actor.apply({'params': params}, jnp.asarray(stack), empty(1)))[0, 0])
        else: u = 0.0
        data.ctrl[:] = u; mujoco.mj_step(model, data)
        up += int(abs(data.qpos[HINGE]) < 0.2)
print(f'CLOSED-LOOP: kept |pole angle| < 0.2 for {up}/{T_eval} steps ({100*up/T_eval:.0f}%) from pixels.')
media.show_video(cl_frames[::2], fps=25)

## Notes
- Trains the real `VisionSkillActor` on real rendered pixels (open-loop corr + closed-loop balance) with no MJWarp dependence.
- Code-correctness gate: `rgb_validation_colab.ipynb` or `pytest tests/test_vision_rgb_smoke.py`.
- To go fuller: use the state-trained NEXUS skills as the teacher (per-skill), same pipeline.